In [1]:
from groq import Groq
import pandas as pd
from sqlalchemy import create_engine, text

# Configure Groq
from dotenv import load_dotenv
import os
load_dotenv('../.env')
client = Groq(api_key=os.environ.get('GROQ_API_KEY'))

# Connect to PostgreSQL
engine = create_engine('postgresql://paulamipaul@localhost:5432/paulamipaul')

print("✓ Groq configured")
print("✓ PostgreSQL connected")
print("✓ Ready to build AI agent")

✓ Groq configured
✓ PostgreSQL connected
✓ Ready to build AI agent


In [2]:
# Define your database schema so the AI knows what tables exist
DB_SCHEMA = """
Tables in the retail database:

1. master_orders - main table with all order details
   columns: order_id, customer_id, order_status, order_purchase_timestamp,
            order_delivered_customer_date, delivery_delay_days,
            processing_time_days, product_id, seller_id, price, freight_value,
            product_category_name_english, product_weight_g,
            customer_city, customer_state, seller_city, seller_state,
            review_score, review_comment_message, payment_value

2. orders_clean - order level data with timestamps
3. products_clean - product details and categories  
4. customers - customer location data
5. sellers - seller location data
6. reviews_clean - customer reviews and scores
7. payments - payment details
8. order_items - individual items per order
"""

def ask_retail_agent(question):
    # Step 1: Ask Gemini to write a SQL query
    prompt = f"""You are a PostgreSQL expert. You have ONE table called master_orders with these columns:
order_id, customer_id, order_status, order_purchase_timestamp,
order_delivered_customer_date, delivery_delay_days, processing_time_days,
product_id, seller_id, price, freight_value,
product_category_name_english, customer_city, customer_state,
seller_city, seller_state, review_score, review_comment_message, payment_value

RULES:
- Use ONLY the master_orders table. No JOINs. No subqueries referencing other tables.
- Return ONLY the SQL query, nothing else.
- Use LIMIT 15.

Question: "{question}"
SQL:"""
    
    sql_response = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[{'role': 'user', 'content': prompt}]
)
    
    # Clean the SQL
    sql = sql_response.choices[0].message.content.strip()
    sql = sql.replace('```sql', '').replace('```', '').strip()
    
    print(f"Question: {question}")
    print(f"\nGenerated SQL:\n{sql}\n")
    
    # Step 2: Run the SQL on PostgreSQL
    try:
        with engine.connect() as conn:
            result = pd.read_sql(text(sql), conn)
        print(f"Results ({len(result)} rows):")
        return result
    except Exception as e:
        print(f"SQL Error: {e}")
        return None

print("✓ AI agent function ready")

✓ AI agent function ready


In [3]:
# Test your agent with a real business question
result = ask_retail_agent("Which product categories have the highest average delivery delay?")
result

Question: Which product categories have the highest average delivery delay?

Generated SQL:
SELECT product_category_name_english, AVG(delivery_delay_days) AS avg_delivery_delay
FROM master_orders
GROUP BY product_category_name_english
ORDER BY avg_delivery_delay DESC
LIMIT 15;

Results (15 rows):


,product_category_name_english,avg_delivery_delay
0,None,NaN
1,arts_and_craftmanship,-6.791667
2,furniture_mattress_and_upholstery,-7.162162
3,home_comfort_2,-8.433333
4,home_confort,-9.858796
5,food,-9.867735
6,audio,-10.140496
7,fashion_underwear_beach,-10.929134
8,electronics,-11.139560
9,books_imported,-11.192982


In [4]:
result2 = ask_retail_agent("Which states have the most orders and what is their average review score?")
result2

Question: Which states have the most orders and what is their average review score?

Generated SQL:
SELECT customer_state, COUNT(order_id) as total_orders, AVG(review_score) as average_review_score
FROM master_orders
GROUP BY customer_state
ORDER BY total_orders DESC
LIMIT 15;

Results (15 rows):


,customer_state,total_orders,average_review_score
0,SP,48093,4.108680
1,RJ,14758,3.793639
2,MG,13301,4.070536
3,RS,6321,4.042249
4,PR,5811,4.087671
5,SC,4216,3.989237
6,BA,3841,3.801954
7,DF,2448,3.991354
8,GO,2366,3.981537
9,ES,2277,3.982628


In [5]:
result3 = ask_retail_agent("Who are the top 10 sellers by total revenue?")
result3

Question: Who are the top 10 sellers by total revenue?

Generated SQL:
SELECT seller_id, SUM(payment_value) AS total_revenue
FROM master_orders
GROUP BY seller_id
ORDER BY total_revenue DESC
LIMIT 10;

Results (10 rows):


,seller_id,total_revenue
0,7c67e1448b00f6e969d365cea6b010ab,512645.19
1,1025f0e2d44d7041d6cf58b6550e0bfa,312456.49
2,4a3ca9315b744ce9f8e9374361493884,306138.80
3,1f50f920176fa81dab994f9023523100,291918.98
4,53243585a1d6dc2643021fd1853d8905,284903.08
5,da8622b14eb17ae2831f4ac5b9dab84a,276578.63
6,4869f7a5dfa277a7dca6462dcf3b52b2,264166.12
7,955fee9216a65b617aa5c0531780ce60,236414.48
8,fa1c13f2614d7b5c4749cbc52fecda94,206513.23
9,7e93a43ef30c4f03f38b393420bc753a,185134.21


In [6]:
result4 = ask_retail_agent("What is the monthly revenue trend over time?")
result4

Question: What is the monthly revenue trend over time?

Generated SQL:
SELECT 
    EXTRACT(YEAR FROM order_purchase_timestamp) AS year,
    EXTRACT(MONTH FROM order_purchase_timestamp) AS month,
    SUM(payment_value) AS monthly_revenue
FROM 
    master_orders
GROUP BY 
    EXTRACT(YEAR FROM order_purchase_timestamp),
    EXTRACT(MONTH FROM order_purchase_timestamp)
ORDER BY 
    year, month
LIMIT 15;

Results (15 rows):


,year,month,monthly_revenue
0,2016.0,9.0,388.47
1,2016.0,10.0,76559.05
2,2016.0,12.0,19.62
3,2017.0,1.0,190806.27
4,2017.0,2.0,351848.13
5,2017.0,3.0,547769.84
6,2017.0,4.0,512126.52
7,2017.0,5.0,737425.31
8,2017.0,6.0,613777.41
9,2017.0,7.0,749242.84


In [7]:
import os
print("GOOGLE_API_KEY:", os.environ.get('GOOGLE_API_KEY'))
print("GOOGLE_APPLICATION_CREDENTIALS:", os.environ.get('GOOGLE_APPLICATION_CREDENTIALS'))

GOOGLE_API_KEY: None
GOOGLE_APPLICATION_CREDENTIALS: None


In [8]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')
key = os.getenv('GEMINI_API_KEY')
print(repr(key))
print(f"Length: {len(key)}")

'AQ.Ab8RN6IFqWpB8Ib19jFjImjqQynHx2diLcfKQ2LhTiIMaIZV7g'
Length: 53
